# Class 1 practical lab — Linux, Python tooling and Git

**Core lab time:** 40 minutes · **Course project:** Engineering Knowledge Assistant

**Environment:** Linux natively; Windows users should run Ubuntu in WSL2; macOS users can use the native terminal. The commands in this lab are intentionally kept to common Linux/macOS shell tools.

**By the end:** inspect the Linux filesystem, understand environment-specific configuration, create a reproducible Python project with `uv`, run quality checks, and make a small Git change.

## Check the development environment

A common source of confusion is having several Python installations. The interpreter used by the notebook, terminal and PyCharm project should be understood and, for project work, aligned with the project's `.venv`.


In [ ]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

print(f"Operating system : {platform.platform()}")
print(f"Python version   : {platform.python_version()}")
print(f"Python executable: {sys.executable}")
print(f"Current folder   : {Path.cwd()}")

for tool in ("git", "uv", "make"):
    print(f"{tool:8} -> {shutil.which(tool) or 'not found by the shell'}")


## 1. Linux filesystem

Linux does not expose each disk as a separate drive letter. It presents one directory tree that starts at **`/` (the filesystem root)**. Other filesystems can be attached to directories in that tree; those directories are *mount points*.

| Directory | Typical purpose |
|---|---|
| `/` | Root of the directory tree |
| `/home` | Users' personal files |
| `/etc` | System-wide configuration |
| `/usr` | User-space programs, libraries and shared data |
| `/var` | Logs and other variable data |
| `/tmp` | Temporary files |
| `/dev` | Device interfaces |
| `/proc`, `/sys` | Virtual filesystems exposing process and kernel information |
| `/mnt`, `/media` | Common mount-point locations |

**Do not confuse `/` with the `root` user.** `/` is a directory; `root` is the administrator account.


In [ ]:
# Inspect the top level of the filesystem. Some entries vary by operating system.
filesystem_root = Path("/")
for entry in sorted(filesystem_root.iterdir(), key=lambda item: item.name.lower()):
    kind = "dir " if entry.is_dir() else "file"
    print(f"{kind:5} /{entry.name}")

print("\nHome directory:", Path.home())
print("/mnt exists     :", Path("/mnt").exists())
print("/proc exists    :", Path("/proc").exists(), "(normally Linux only)")


### Try these commands in a terminal. What do each of them do?

```bash
pwd          
ls -la /      
cd ~/        
mkdir -p ~/projects/class1-lab
```

**Root privileges:** ordinary work should happen as your own user. `sudo` runs one command with elevated privileges; use it only when an administrative operation genuinely requires it. Do not use `sudo` to install project Python dependencies.


## 2. Environment variables

The same application may run in development, testing and production. Environment variables let the deployment environment provide configuration without editing Python source code.

Examples:

- `APP_ENV=DEV`, `APP_ENV=TEST`, `APP_ENV=PROD`
- `LOG_LEVEL=DEBUG` or `LOG_LEVEL=INFO`
- service URLs and feature flags

Secrets should be injected securely and must never be committed to Git. A local `.env` file should be ignored by Git if it contains secrets.


In [ ]:
# Read configuration with safe defaults. These values are illustrative, not secrets.
app_env = os.getenv("APP_ENV", "DEV").upper()
log_level = os.getenv("LOG_LEVEL", "DEBUG" if app_env == "DEV" else "INFO").upper()

settings = {
    "app_env": app_env,
    "log_level": log_level,
    "debug": app_env == "DEV",
}
print(settings)
# Set ENV variables with different values and run again.

## 3. Python projects and `uv`

Installing every dependency globally is fragile: two projects can require incompatible versions of the same library. A project-local `.venv` isolates those dependencies. The project manifest and lockfile make the environment reproducible for teammates and CI.

- `pyproject.toml`: project metadata, Python requirement, dependencies and tool configuration.
- `uv.lock`: exact resolved versions, including transitive dependencies.
- `.venv/`: local environment; do **not** commit it.

`uv` manages Python versions, virtual environments, dependencies and lockfiles. The practical commands are:

```bash
uv venv                         # create a python virtual environment
uv sync                         # create/synchronize from project files
uv add httpx                    # add a runtime dependency
uv add --dev ruff mypy          # add development tools
uv add --group test pytest      # add a test dependency
```

Use `uv sync` to install an existing project's requirements; use `uv add` when changing its dependencies. Commit both `pyproject.toml` and `uv.lock`.


In [ ]:
# pyproject.toml is TOML: Python 3.11+ can parse it with the standard library.
import tomllib

pyproject_example = """[project]
name = "engineering-knowledge-assistant"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = []

[dependency-groups]
dev = ["ruff", "mypy", "pre-commit"]
test = ["pytest", "pytest-cov"]

[tool.ruff]
line-length = 88

[tool.mypy]
python_version = "3.12"
strict = true
"""

config = tomllib.loads(pyproject_example)
print("Project:", config["project"]["name"])
print("Python requirement:", config["project"]["requires-python"])
print("Dev tools:", config["dependency-groups"]["dev"])
print("Test tools:", config["dependency-groups"]["test"])


### Create a disposable lab project

For this guided exercise, the next cell creates a small python project so we can inspect its files and Makefile.

The cell will **not overwrite** an existing lab folder. If you have already created one, use it or rename it before re-running this cell.


In [ ]:
LAB_ROOT = Path.cwd() / "class1-lab-project"

if LAB_ROOT.exists():
    print(f"Already exists: {LAB_ROOT} (leaving existing files untouched)")
else:
    (LAB_ROOT / "src").mkdir(parents=True)
    (LAB_ROOT / "tests").mkdir()
    (LAB_ROOT / "examples").mkdir()

    lab_pyproject = """[project]
name = "class1-lab-project"
version = "0.1.0"
requires-python = ">=3.12"
dependencies = []

[dependency-groups]
dev = ["ruff", "mypy", "pre-commit"]
test = ["pytest", "pytest-cov"]

[tool.uv]
package = false

[tool.ruff]
line-length = 88
target-version = "py312"
src = ["src"]

[tool.ruff.lint]
select = ["E", "F", "I", "B", "UP"]

[tool.mypy]
python_version = "3.12"
mypy_path = "src"
strict = true

[tool.pytest.ini_options]
testpaths = ["tests"]
pythonpath = ["src"]
"""
    (LAB_ROOT / "pyproject.toml").write_text(lab_pyproject, encoding="utf-8")
    (LAB_ROOT / "Makefile").write_text("export PYTHONPATH=$(CURDIR)/src\n\nall: lint test\n\nclean:\n\trm -f .coverage\n\trm -rf .mypy_cache\n\tfind . -type d -name \"__pycache__\" -exec rm -rf {} +\n\nformat:\n\tuv run ruff format src tests\n\tuv run ruff check src tests --select I --fix  # optimizes imports\n\ninstall:\n\tuv sync\n\ninstall-test: install\n\tuv sync --group test\n\nlint: lint-ruff lint-mypy\n\nlint-ruff:\n\tuv run ruff check src\n\tuv run ruff format --check src\n\nlint-mypy:\n\tuv run mypy src\n\tuv run mypy tests\n\ntest:\n\tuv run pytest tests --cov src\n", encoding="utf-8")
    (LAB_ROOT / ".gitignore").write_text(
        ".venv/\n__pycache__/\n.pytest_cache/\n.mypy_cache/\n.ruff_cache/\n.coverage\n.env\n",
        encoding="utf-8",
    )
    (LAB_ROOT / "src" / "lab_demo.py").write_text(
        "def describe_file(name: str, size_bytes: int) -> str:\n"
        "    return f'{name}: {size_bytes} bytes'\n",
        encoding="utf-8",
    )
    (LAB_ROOT / "tests" / "test_lab_demo.py").write_text(
        "from lab_demo import describe_file\n\n"
        "def test_describe_file() -> None:\n"
        "    assert describe_file('manual.txt', 5) == 'manual.txt: 5 bytes'\n",
        encoding="utf-8",
    )
    print(f"Created lab project: {LAB_ROOT}")

print("\nFiles:")
if LAB_ROOT.exists():
    for path in sorted(LAB_ROOT.rglob("*")):
        if path.is_file() and ".git" not in path.parts:
            print(" -", path.relative_to(LAB_ROOT).as_posix())


### Install and run the project checks

From a terminal at the generated project root:

```bash
make install-test
make lint
make format
make lint
make test
```

`make install-test` runs the Makefile's `uv sync` commands for the default development tools and the test group. The first run needs network access to resolve/download dependencies. `make format` changes files; `make lint` checks code; `make test` checks behavior.


## 4. mypy: catch a type mismatch before runtime

Python type annotations do not enforce types when a program runs. A static checker such as mypy can still detect that a call violates the function's declared contract.

The following example is valid Python at runtime, but the call is inconsistent with the annotation:


In [ ]:
def shipping_cost(weight_kg: float) -> float:
    return weight_kg * 2

# Python executes this and repeats the string; it does not enforce the annotation.
print(shipping_cost("heavy"))  # Runtime output: heavyheavy

mypy_demo_source = "\n".join([
    "def shipping_cost(weight_kg: float) -> float:",
    "    return weight_kg * 2",
    "",
    "print(shipping_cost('heavy'))",
    "",
])
example_dir = LAB_ROOT / "examples"
example_dir.mkdir(exist_ok=True)
(example_dir / "mypy_demo.py").write_text(mypy_demo_source, encoding="utf-8")
print("\nSaved runnable demo:", example_dir / "mypy_demo.py")

# A runnable example file is created under examples/ in the lab project.
# From the lab project root, run:
# python examples/mypy_demo.py
# mypy examples/mypy_demo.py

# Python runs it; mypy reports that "heavy" is not a float:
# examples/mypy_demo.py:4: error: Argument 1 to "shipping_cost" has incompatible type "str"; expected "float"  [arg-type]


### Ruff: formatting, linting and imports

Ruff is written in Rust and combines several Python-quality tasks. In this project we use:

- `ruff format` for consistent formatting;
- `ruff check` for lint rules;
- `ruff check --select I --fix` for import sorting.

The Makefile puts the common workflow behind `make format` and `make lint`. 

The Ruff documentation has benchmark charts and explains the supported rules: <https://docs.astral.sh/ruff/>.

In [ ]:
# Example of deliberately messy code to save as src/ruff_demo.py:
ruff_example = "import os\nimport json\n\ndef summarize( values:dict[str,int])->str:\n    prefix=os.getenv(\"PREFIX\", \"\")\n    return prefix+json.dumps(values,sort_keys=True)\n"

ruff_demo_path = LAB_ROOT / "src" / "ruff_demo.py"
ruff_demo_path.write_text(ruff_example, encoding="utf-8")
print("Saved deliberately unformatted code to:", ruff_demo_path)
print(ruff_example)
print("\nRun from the lab project:")
print("ruff format src tests")
print("ruff check src tests --select I --fix")
print("make lint")

## 5. Git and GitHub: 

Git is distributed version control: each clone has local history, branches and commits. 

GitHub hosts a remote repository and adds collaboration features such as pull requests and reviews.

In the lab project, inspect the working tree and stage the files:


In [ ]:
if shutil.which("git"):
    if not (LAB_ROOT / ".git").exists():
        subprocess.run(["git", "init", "-b", "main"], cwd=LAB_ROOT, check=True)
    commands = [
        ["git", "status", "--short"],
        ["git", "add", "pyproject.toml", "Makefile", ".gitignore", "src", "tests"],
        ["git", "status", "--short"],
        ["git", "diff", "--cached", "--stat"],
    ]
    for command in commands:
        result = subprocess.run(command, cwd=LAB_ROOT, text=True, capture_output=True, check=False)
        print(f"$ {' '.join(command)}")
        print(result.stdout or result.stderr)
else:
    print("Git is not available. Install Git before continuing.")

### Commit conventions and hooks

A Conventional Commit header follows:

```text
<type>(<optional-scope>): <description>
```

Examples: `feat(catalog): discover PDF files`, `fix(catalog): handle missing folders`, `test(catalog): cover mixed-case extensions`.

The homework repository includes `pre-commit` hooks for Ruff, mypy and commit-message validation. After installing its dependencies, run:

```bash
pre-commit install
pre-commit run --all-files
```

For a GitHub remote, use the HTTPS URL supplied for the repository, then push the branch and open a pull request. Use HTTPS for the remote in this class.

### Try a small commit

Configure your identity locally for this repository, then make a commit. Replace the example values with your own:

```bash
git config user.name "Your Name"
git config user.email "you@example.org"
git commit -m "chore: initialize class 1 lab project"
```

How to connect to a GitHub repository?

```bash
# via https:
git clone https://github.com/<user>/<repo>.git

# via ssh 
git clone git@github.com:/<user>/<repo>.git
```

## Wrap-up

Before leaving, check that you can explain:

1. What `/` means and why `/home`, `/etc`, `/var` and mount points exist.
2. Why `sudo` should be used sparingly and why environment variables separate configuration from code.
3. What `pyproject.toml`, `uv.lock` and `.venv/` each do.
4. The difference between Ruff and mypy.
5. The difference between `git add`, `git commit` and `git push`.

# **Homework:** 
Clone the Engineering Knowledge Assistant starter project, complete the TODOs, keep the checks green, and submit a pull request.